# California GTFS Shared-Stop Toolkit: ArcGIS Pro runner

Run the toolkit from ArcGIS Pro.

1. Edit the paths and settings in the **USER SETTINGS** section of the code cell below.
2. Run the cell.

The toolkit can be a ZIP downloaded from GitHub or a folder (for example a git clone). A ZIP is extracted to a short local folder first, which avoids Windows path-length problems.

With `offline_context_only = True` (the default) the run uses only cached served shapes and road centerlines and never queries BigQuery or the road service. If a cache is missing, the run stops and says which one.


In [ ]:
from pathlib import Path
import importlib
import shutil
import sys
import zipfile

# ============================================================================
# USER SETTINGS (edit these)
# ============================================================================

# Where the toolkit is. Either works:
#   - the ZIP downloaded from GitHub ("Download ZIP"), or
#   - a folder, such as a git clone of the repository.
toolkit_location = Path(
    r"C:\path\to\california_gtfs_shared_stop_toolkit.zip"
)

# Folder containing the warehouse CSV inputs and the served-shape caches.
warehouse_dir = Path(
    r"C:\path\to\gtfs_warehouse"
)

# Geodatabase that already holds the Caltrans All Roads cache for this
# warehouse run (Caltrans_All_Roads_GTFS_Context_<run_stamp>). The cache is
# copied into this run's analysis.gdb so the road service is not queried.
# Leave as None if you do not have one.
road_cache_gdb = None

# True = use only the cached served shapes and road centerlines. The run stops
# with a clear message if a cache is missing, so it can never query BigQuery
# (which costs money) or the road service by surprise.
# False = use the caches when they exist and query the services when they don't.
offline_context_only = True

# Add the two final QA layers to the active ArcGIS Pro map.
add_to_map = True

# Normally keep False. Set True only when you intend to replace an existing
# output folder that was produced by different toolkit code.
replace_existing_outputs = False

# A ZIP is extracted here first. A short local path avoids Windows path-length
# problems with long OneDrive or Downloads paths. Not used for a folder.
runtime_dir = Path.home() / "_gtfs_shared_stop_runtime"

# ============================================================================
# FIND THE TOOLKIT
# ============================================================================

print("Toolkit location:", toolkit_location)
print("Warehouse:", warehouse_dir)
print("Road cache geodatabase:", road_cache_gdb)
print("Offline (cached context only):", offline_context_only)

if not toolkit_location.exists():
    raise FileNotFoundError(f"Toolkit ZIP or folder not found: {toolkit_location}")

if not warehouse_dir.exists():
    raise FileNotFoundError(f"Warehouse folder not found: {warehouse_dir}")

if road_cache_gdb is not None and not Path(road_cache_gdb).exists():
    raise FileNotFoundError(f"Road cache geodatabase not found: {road_cache_gdb}")

if toolkit_location.is_dir():
    search_root = toolkit_location
else:
    # Extract a fresh copy so an older extracted copy is never used by mistake.
    if runtime_dir.exists():
        shutil.rmtree(runtime_dir)
    runtime_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(toolkit_location, "r") as zf:
        zf.extractall(runtime_dir)
    search_root = runtime_dir
    print("Extracted to:", runtime_dir)

# The package folder is either at the top level or one folder down
# (a GitHub ZIP contains a single folder such as <repository>-main).
package_root = None
if (search_root / "shared_stop_toolkit").exists():
    package_root = search_root
else:
    for p in search_root.iterdir():
        if p.is_dir() and (p / "shared_stop_toolkit").exists():
            package_root = p
            break

if package_root is None:
    raise FileNotFoundError(
        f"Could not find the shared_stop_toolkit folder in {toolkit_location}"
    )

# Forget any copy of the toolkit already loaded in this notebook session, so
# this run always uses the code found above.
for module_name in list(sys.modules):
    if module_name == "shared_stop_toolkit" or module_name.startswith("shared_stop_toolkit."):
        del sys.modules[module_name]

if str(package_root) in sys.path:
    sys.path.remove(str(package_root))
sys.path.insert(0, str(package_root))
importlib.invalidate_caches()

from shared_stop_toolkit.workflow import run_shared_stop_analysis

print("Toolkit loaded from:", package_root)

# ============================================================================
# RUN THE TOOLKIT
# ============================================================================

result = run_shared_stop_analysis(
    warehouse_dir=warehouse_dir,
    add_to_map=add_to_map,
    offline_context_only=offline_context_only,
    road_cache_gdb=road_cache_gdb,
    replace_existing_outputs=replace_existing_outputs,
)

# ============================================================================
# RUN SUMMARY
# ============================================================================

print("\n" + "=" * 72)
print("SHARED-STOP ANALYSIS COMPLETE")
print("=" * 72)
print("Output folder:", result.get("output_dir"))
print("Source run stamp:", result.get("source_run_stamp"))
print("Pair rows scored:", result.get("pair_rows_scored"))
print("Shared-stop groups:", result.get("shared_stop_group_count"))
print("High-confidence groups:", result.get("high_confidence_group_count"))
print("Review groups:", result.get("review_group_count"))
print("Resolved recommendation locations:", result.get("resolved_recommendation_location_count"))
print("Unresolved locations:", result.get("unresolved_location_count"))
print("Stop-location issues:", result.get("stop_location_issue_count"))
print("Shared-stop candidates CSV:", result.get("shared_stop_candidates_csv"))
print("Stop-location issues CSV:", result.get("stop_location_issues_over_25ft_csv"))
print("Run manifest:", result.get("run_manifest"))
